# WMCP — World Model Communication Protocol Demo

**Discrete compositional communication between heterogeneous vision foundation models.**

This notebook demonstrates the core WMCP capabilities:
1. Train heterogeneous agents (V-JEPA 2 + DINOv2) to communicate about physics
2. Onboard a new encoder (CLIP ViT-L/14) into the existing protocol
3. Validate compositionality with triple metrics (PosDis, TopSim, BosDis)
4. Run the compliance suite

**Runtime:** ~3 minutes on free Colab CPU.

Paper: [doi:10.5281/zenodo.19197757](https://doi.org/10.5281/zenodo.19197757) · Code: [github.com/TomekKaszynski/emergent-physics-comm](https://github.com/TomekKaszynski/emergent-physics-comm)

## 1. Setup

Install the wmcp package and dependencies. No GPU required.

In [ ]:
!pip install -q torch numpy scipy

# Install wmcp from the repo (or pip install wmcp if published)
# For this demo, we inline the core modules so the notebook is self-contained.

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
from scipy import stats

print(f"PyTorch: {torch.__version__}")
print(f"Device: CPU (no GPU required)")
print(f"WMCP v0.1.0")

## 2. Core Architecture

The WMCP protocol stack: frozen encoder → projection layer → Gumbel-Softmax bottleneck → discrete symbols.

The projection layer is the **only trainable component** per encoder (~400K parameters).

In [ ]:
# ═══ WMCP Core Architecture (self-contained) ═══

class ProjectionLayer(nn.Module):
    """Trainable projection from frozen encoder features to bottleneck."""
    def __init__(self, input_dim, hidden_dim=128, n_frames=4):
        super().__init__()
        ks = min(3, max(1, n_frames))
        self.temporal = nn.Sequential(
            nn.Conv1d(input_dim, 256, ks, padding=ks//2), nn.ReLU(),
            nn.Conv1d(256, 128, ks, padding=ks//2), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1))
        self.fc = nn.Sequential(nn.Linear(128, hidden_dim), nn.ReLU())
    def forward(self, x):
        return self.fc(self.temporal(x.permute(0, 2, 1)).squeeze(-1))

class GumbelSoftmaxBottleneck(nn.Module):
    """Discrete bottleneck: continuous logits → one-hot symbols."""
    def __init__(self, vocab_size=3, n_heads=2, hidden_dim=128):
        super().__init__()
        self.vocab_size = vocab_size
        self.heads = nn.ModuleList([nn.Linear(hidden_dim, vocab_size) for _ in range(n_heads)])
    def forward(self, h, tau=1.0, hard=True):
        msgs, logits = [], []
        for head in self.heads:
            l = head(h)
            m = F.gumbel_softmax(l, tau=tau, hard=hard) if self.training else F.one_hot(l.argmax(-1), self.vocab_size).float()
            msgs.append(m); logits.append(l)
        return torch.cat(msgs, -1), logits

class AgentSender(nn.Module):
    """Single agent: projection + bottleneck."""
    def __init__(self, input_dim, hidden_dim=128, vocab_size=3, n_heads=2, n_frames=4):
        super().__init__()
        self.projection = ProjectionLayer(input_dim, hidden_dim, n_frames)
        self.bottleneck = GumbelSoftmaxBottleneck(vocab_size, n_heads, hidden_dim)
    def forward(self, x, tau=1.0, hard=True):
        return self.bottleneck(self.projection(x), tau, hard)

class Protocol(nn.Module):
    """Multi-agent compositional communication protocol."""
    def __init__(self, agent_configs, hidden_dim=128, vocab_size=3, n_heads=2):
        super().__init__()
        self.vocab_size = vocab_size; self.n_heads = n_heads
        self.n_agents = len(agent_configs)
        self.msg_dim = self.n_agents * n_heads * vocab_size
        self.senders = nn.ModuleList([
            AgentSender(dim, hidden_dim, vocab_size, n_heads, nf)
            for dim, nf in agent_configs])
        self.receiver = nn.Sequential(
            nn.Linear(self.msg_dim * 2, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1))

    def encode(self, views, tau=1.0, hard=True):
        msgs, logits = [], []
        for s, v in zip(self.senders, views):
            m, l = s(v, tau, hard); msgs.append(m); logits.extend(l)
        return torch.cat(msgs, -1), logits

    def communicate(self, va, vb, tau=1.0, hard=True):
        ma, _ = self.encode(va, tau, hard)
        mb, _ = self.encode(vb, tau, hard)
        return self.receiver(torch.cat([ma, mb], -1)).squeeze(-1)

print("✓ Architecture defined")
print(f"  Protocol with 2 agents: {sum(p.numel() for p in Protocol([(1024,4),(384,4)]).parameters()):,} parameters")

## 3. Simulate Heterogeneous Communication

We simulate two agents with **different encoder architectures** (V-JEPA 2 at 1024-dim, DINOv2 at 384-dim) communicating about physical properties through the discrete bottleneck.

For this demo, we use synthetic features that mimic the structure of real encoder outputs — features are correlated with a hidden "mass" property that agents must learn to communicate about.

In [ ]:
# Generate synthetic physics features
# Features are correlated with a hidden "mass" property
N = 200  # number of scenes
n_frames = 8
rng = np.random.RandomState(42)

# Hidden physical property: mass (continuous)
mass = rng.uniform(1, 100, N)
obj_names = [f"obj_{i % 20}" for i in range(N)]

# Simulate V-JEPA 2 features (1024-dim, temporal)
# Features embed mass information in a nonlinear way
vjepa_base = rng.randn(N, 1024).astype(np.float32)
for i in range(N):
    vjepa_base[i, :50] += mass[i] * 0.1  # mass signal in first 50 dims
vjepa_feat = torch.from_numpy(
    np.stack([vjepa_base + rng.randn(N, 1024).astype(np.float32) * 0.1
              for _ in range(n_frames)], axis=1))  # (N, 8, 1024)

# Simulate DINOv2 features (384-dim, static — replicated across time)
dino_base = rng.randn(N, 384).astype(np.float32)
for i in range(N):
    dino_base[i, :20] += mass[i] * 0.15  # mass signal in first 20 dims
dino_feat = torch.from_numpy(dino_base).unsqueeze(1).expand(-1, n_frames, -1).contiguous()

fpa = n_frames // 2  # frames per agent
print(f"✓ Synthetic features generated")
print(f"  V-JEPA 2: {vjepa_feat.shape} (temporal, 1024-dim)")
print(f"  DINOv2:   {dino_feat.shape} (static replicated, 384-dim)")
print(f"  Mass range: {mass.min():.1f} – {mass.max():.1f}")
print(f"  {N} scenes, {len(set(obj_names))} unique objects")

In [ ]:
# Train heterogeneous protocol (V-JEPA + DINOv2)
protocol = Protocol([(1024, fpa), (384, fpa)], vocab_size=3, n_heads=2)
optimizer = torch.optim.Adam(protocol.parameters(), lr=1e-3)

agent_views = [vjepa_feat[:, :fpa, :], dino_feat[:, fpa:, :]]
mass_t = torch.tensor(mass, dtype=torch.float32)

# Object-level holdout
unique_objs = sorted(set(obj_names))
holdout = set(rng.choice(unique_objs, max(4, len(unique_objs)//5), replace=False))
train_ids = np.array([i for i, o in enumerate(obj_names) if o not in holdout])
holdout_ids = np.array([i for i, o in enumerate(obj_names) if o in holdout])

print("Training heterogeneous protocol (V-JEPA 2 + DINOv2)...")
t0 = time.time()
best_acc = 0

for ep in range(200):
    protocol.train()
    tau = 3.0 + (1.0 - 3.0) * ep / 199
    hard = ep >= 30

    # Sample pairs
    ia = rng.choice(train_ids, 32); ib = rng.choice(train_ids, 32)
    s = ia == ib
    while s.any(): ib[s] = rng.choice(train_ids, s.sum()); s = ia == ib
    md = np.abs(mass[ia] - mass[ib]); keep = md > 0.5
    if keep.sum() < 4: continue
    ia, ib = ia[keep], ib[keep]

    va = [v[ia] for v in agent_views]; vb = [v[ib] for v in agent_views]
    label = (mass_t[ia] > mass_t[ib]).float()
    pred = protocol.communicate(va, vb, tau, hard)
    loss = F.binary_cross_entropy_with_logits(pred, label)

    optimizer.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(protocol.parameters(), 1.0)
    optimizer.step()

    if (ep + 1) % 50 == 0:
        protocol.eval()
        with torch.no_grad():
            c = t = 0
            for _ in range(20):
                ia_h = rng.choice(holdout_ids, min(16, len(holdout_ids)))
                ib_h = rng.choice(holdout_ids, min(16, len(holdout_ids)))
                mdh = np.abs(mass[ia_h] - mass[ib_h]); kh = mdh > 0.5
                if kh.sum() < 2: continue
                ia_h, ib_h = ia_h[kh], ib_h[kh]
                p = protocol.communicate([v[ia_h] for v in agent_views],
                                          [v[ib_h] for v in agent_views]) > 0
                c += (p == (mass_t[ia_h] > mass_t[ib_h])).sum().item()
                t += len(ia_h)
            acc = c / max(t, 1)
            best_acc = max(best_acc, acc)
        print(f"  Epoch {ep+1:3d}: loss={loss.item():.3f}  holdout_acc={acc:.1%}  best={best_acc:.1%}")

elapsed = time.time() - t0
print(f"\n✓ Training complete in {elapsed:.1f}s")
print(f"  Best holdout accuracy: {best_acc:.1%}")

## 4. Compositionality Metrics

Verify the protocol developed compositional structure using three independent metrics:
- **PosDis**: Does each message position specialize for one property?
- **TopSim**: Do similar meanings produce similar messages?
- **BosDis**: Do specific symbols encode specific attributes?

In [ ]:
# ═══ Metrics (self-contained) ═══

def mutual_information(x, y):
    n = len(x); mi = 0.0
    for xv in np.unique(x):
        for yv in np.unique(y):
            pxy = np.sum((x==xv)&(y==yv))/n; px = np.sum(x==xv)/n; py = np.sum(y==yv)/n
            if pxy > 0 and px > 0 and py > 0: mi += pxy * np.log(pxy/(px*py))
    return mi

def compute_posdis(tokens, attributes, vocab_size):
    n_pos, n_attr = tokens.shape[1], attributes.shape[1]
    mi = np.zeros((n_pos, n_attr))
    for p in range(n_pos):
        for a in range(n_attr):
            mi[p, a] = mutual_information(tokens[:, p], attributes[:, a])
    pd = 0.0
    for p in range(n_pos):
        s = np.sort(mi[p])[::-1]
        if s[0] > 1e-10: pd += (s[0] - s[1]) / s[0]
    return pd / n_pos if n_pos > 0 else 0, mi

def compute_topsim(tokens, p1, p2, n_pairs=5000):
    rng2 = np.random.RandomState(42); n = len(tokens)
    md, msgd = [], []
    for _ in range(min(n_pairs, n*(n-1)//2)):
        i, j = rng2.choice(n, 2, replace=False)
        md.append(abs(int(p1[i])-int(p1[j])) + abs(int(p2[i])-int(p2[j])))
        msgd.append(int((tokens[i] != tokens[j]).sum()))
    r, _ = stats.spearmanr(md, msgd)
    return 0.0 if np.isnan(r) else float(r)

def compute_bosdis(tokens, attributes, vocab_size):
    n_attr = attributes.shape[1]; bd = 0.0; na = 0
    for s in range(vocab_size):
        cs = np.any(tokens == s, axis=1).astype(int)
        if cs.sum() == 0 or cs.sum() == len(tokens): continue
        mis = [mutual_information(cs, attributes[:, a]) for a in range(n_attr)]
        sm = sorted(mis, reverse=True)
        if sm[0] > 1e-10:
            bd += (sm[0] - sm[1]) / sm[0] if len(sm) > 1 and sm[1] > 1e-10 else 1.0
            na += 1
    return bd / max(na, 1)

# Extract tokens
protocol.eval()
with torch.no_grad():
    _, logits = protocol.encode(agent_views)
    tokens = np.stack([l.argmax(-1).numpy() for l in logits], axis=1)

# Build attributes
mass_bins = np.digitize(mass, np.quantile(mass, [0.2, 0.4, 0.6, 0.8]))
uo = sorted(set(obj_names)); oi = {o: i for i, o in enumerate(uo)}
obj_bins = np.digitize(np.array([oi[o] for o in obj_names]),
                        np.quantile(np.arange(len(uo)), [0.2, 0.4, 0.6, 0.8]))
attributes = np.stack([mass_bins, obj_bins], axis=1)

posdis, mi_matrix = compute_posdis(tokens, attributes, 3)
topsim = compute_topsim(tokens, mass_bins, obj_bins)
bosdis = compute_bosdis(tokens, attributes, 3)

print("╔══════════════════════════════════╗")
print("║  COMPOSITIONALITY METRICS        ║")
print("╠══════════════════════════════════╣")
print(f"║  PosDis:  {posdis:.3f}  (threshold: >0.5)  ║")
print(f"║  TopSim:  {topsim:.3f}                    ║")
print(f"║  BosDis:  {bosdis:.3f}                    ║")
print("╚══════════════════════════════════╝")
print(f"\nMI matrix (position × attribute):")
print(f"  {'':>6s}  mass   object")
for p in range(mi_matrix.shape[0]):
    print(f"  pos {p}: {mi_matrix[p,0]:.3f}  {mi_matrix[p,1]:.3f}")

## 5. Onboard a New Encoder (CLIP ViT-L/14)

The protocol is already trained with V-JEPA 2 + DINOv2. Now we add a **third architecture** (CLIP, 768-dim) by freezing the existing agents and fine-tuning only the new CLIP projection layer.

This should converge in ~50 training steps (Phase 104 result).

In [ ]:
# Simulate CLIP features (768-dim, static)
clip_base = rng.randn(N, 768).astype(np.float32)
for i in range(N):
    clip_base[i, :30] += mass[i] * 0.12
clip_feat = torch.from_numpy(clip_base).unsqueeze(1).expand(-1, n_frames, -1).contiguous()

# Replace agent 1 (DINOv2) with CLIP
base_acc = best_acc
print(f"Base protocol accuracy: {base_acc:.1%}")
print(f"Target (90% of base): {base_acc * 0.9:.1%}\n")

# Create new CLIP sender
torch.manual_seed(5000)
clip_sender = AgentSender(768, 128, 3, 2, fpa)
protocol.senders[1] = clip_sender

# Freeze everything except CLIP sender
for name, param in protocol.named_parameters():
    param.requires_grad = "senders.1" in name

agent_views_new = [vjepa_feat[:, :fpa, :], clip_feat[:, fpa:, :]]
ft_opt = torch.optim.Adam(clip_sender.parameters(), lr=1e-3)

print("Onboarding CLIP into existing protocol...")
print(f"  {'Step':>6s} │ {'Accuracy':>10s} │ Status")
print(f"  {'─'*6}─┼─{'─'*10}─┼─{'─'*20}")

for step in range(100):
    protocol.train()
    ia = rng.choice(train_ids, 32); ib = rng.choice(train_ids, 32)
    s = ia == ib
    while s.any(): ib[s] = rng.choice(train_ids, s.sum()); s = ia == ib
    md = np.abs(mass[ia] - mass[ib]); keep = md > 0.5
    if keep.sum() < 4: continue
    ia, ib = ia[keep], ib[keep]
    va = [v[ia] for v in agent_views_new]; vb = [v[ib] for v in agent_views_new]
    label = (mass_t[ia] > mass_t[ib]).float()
    pred = protocol.communicate(va, vb)
    loss = F.binary_cross_entropy_with_logits(pred, label)
    ft_opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(clip_sender.parameters(), 1.0)
    ft_opt.step()

    if (step + 1) % 10 == 0:
        protocol.eval()
        with torch.no_grad():
            c = t = 0
            for _ in range(20):
                ia_h = rng.choice(holdout_ids, min(16, len(holdout_ids)))
                ib_h = rng.choice(holdout_ids, min(16, len(holdout_ids)))
                mdh = np.abs(mass[ia_h] - mass[ib_h]); kh = mdh > 0.5
                if kh.sum() < 2: continue
                ia_h, ib_h = ia_h[kh], ib_h[kh]
                p = protocol.communicate([v[ia_h] for v in agent_views_new],
                                          [v[ib_h] for v in agent_views_new]) > 0
                c += (p == (mass_t[ia_h] > mass_t[ib_h])).sum().item(); t += len(ia_h)
            acc = c / max(t, 1)
        status = "← CONVERGED (90% of base)" if acc >= base_acc * 0.9 else ""
        print(f"  {step+1:6d} │ {acc*100:9.1f}% │ {status}")

print(f"\n✓ CLIP onboarded into existing V-JEPA+DINOv2 protocol")

## 6. Latency Benchmark

How fast is a single communication round?

In [ ]:
protocol.eval()
# Warmup
for _ in range(10):
    with torch.no_grad():
        protocol.communicate([v[:1] for v in agent_views_new],
                              [v[1:2] for v in agent_views_new])

latencies = []
for _ in range(500):
    i, j = rng.randint(0, N), rng.randint(0, N)
    t_s = time.perf_counter()
    with torch.no_grad():
        protocol.communicate([v[i:i+1] for v in agent_views_new],
                              [v[j:j+1] for v in agent_views_new])
    latencies.append((time.perf_counter() - t_s) * 1000)

lats = np.array(latencies)
print(f"Latency (500 iterations, CPU):")
print(f"  Mean:       {np.mean(lats):.2f}ms")
print(f"  Median:     {np.median(lats):.2f}ms")
print(f"  P95:        {np.percentile(lats, 95):.2f}ms")
print(f"  Throughput: {1000/np.mean(lats):.0f} comms/s")
print(f"\n  Real-time viable (<10ms): {'YES ✓' if np.mean(lats) < 10 else 'NO'}")

## Summary

This demo showed:
1. **Heterogeneous communication** — V-JEPA 2 (1024-dim) and DINOv2 (384-dim) agents develop shared compositional protocols through a K=3 discrete bottleneck
2. **Compositionality** — verified by three independent metrics (PosDis, TopSim, BosDis)
3. **Fast onboarding** — CLIP joins the protocol in ~50 steps by fine-tuning only its projection layer
4. **Sub-10ms latency** — suitable for real-time robotics

For real encoder features (not synthetic), see the full experiment codebase:
[github.com/TomekKaszynski/emergent-physics-comm](https://github.com/TomekKaszynski/emergent-physics-comm)

Full protocol specification: [protocol-spec/](https://github.com/TomekKaszynski/emergent-physics-comm/tree/main/protocol-spec)